In [1]:
import pandas as pd
from datetime import datetime

import sys
from pathlib import Path

In [3]:
sys.path.append(r"C:/Users/sanya/structured-products-analytics")

from src.reverse_convertible import ReverseConvertible
from src.scenario_engine import ScenarioEngine
from src.portfolio_analytics import PortfolioAnalytics

### 🔹 Input Data — Portfolio Definition

This section defines a sample portfolio of structured products used throughout the analysis.

Each row represents a single product and contains all relevant contractual and market information required for valuation, including:

- product identifiers (product_id, product_type, style)  
- position details (position_units, notional, cost_price, currency)  
- underlying assets and identifiers (tickers, ISINs)  
- payoff parameters (strike levels, barrier level as a percentage of strike, coupon)  
- market data (initial fixing levels and current spot prices)  
- lifecycle information (initial fixing date, maturity date)    

Multi-asset products are represented using lists to capture all underlying components consistently.

The dataset is structured to resemble real trading desk inputs, allowing the analytics engine to scale naturally from single-product evaluation to portfolio-level risk analysis.

In [4]:
p1 = {
    "product_id": "CH1483491150",
    "product_type": "BRC",
    "type_style": "European",
    "underlyings": ["ALCON"],
    "underlying_isins": ["CH0432492467"],
    "currency": "CHF",
    "position_units": 10,
    "notional": 1000,
    "cost_price": 1.00,
    "initial_levels": [59.72],
    "current_spots": [58.76],
    "strike": [59.72],
    "barrier_pct": 0.70,
    "coupon": 0.04,
    "initial_fixing_date": "2025-11-10",
    "maturity_date": "2026-11-17",
    "barrier_breached": False
}

p2 = {
    "product_id": "CH1449111066",
    "product_type": "MBRC",
    "type_style": "European",
    "underlyings": ["ABB", "HOLCIM", "NOVARTIS", "ROCHE"],
    "underlying_isins": [
        "CH0012221716",
        "CH0012214059",
        "CH0012005267",
        "CH0012032048"
    ],
    "currency": "CHF",
    "position_units": 5,
    "notional": 1000,
    "cost_price": 0.98,
    "initial_levels": [35.00, 70.00, 90.00, 250.00],
    "current_spots": [34.00, 68.00, 92.00, 245.00],
    "strike": [35.00, 70.00, 90.00, 250.00],
    "barrier_pct": 0.70,
    "coupon": 0.0675,
    "initial_fixing_date": "2025-12-30",
    "maturity_date": "2026-12-28",
    "barrier_breached": True
}

p3 = {
    "product_id": "CH1461018793",
    "product_type": "MBRC",
    "type_style": "European",
    "underlyings": ["ABB", "LONZA", "NESTLE"],
    "underlying_isins": ["CH0012221716", "CH0013841017", "CH0038863350"],
    "currency": "CHF",
    "position_units": 1,
    "notional": 10000,
    "cost_price": 1.00,
    "initial_levels": [53.94, 555.20, 72.49],
    "current_spots": [53.94, 555.20, 72.49],
    "strike": [53.94, 555.20, 72.49],
    "barrier_pct": 0.70,
    "coupon": 0.0866,
    "initial_fixing_date": "2025-08-19",
    "maturity_date": "2026-08-19",
    "barrier_breached": False
}

portfolio = pd.DataFrame([p1, p2, p3])
portfolio

,product_id,product_type,type_style,underlyings,underlying_isins,currency,position_units,notional,cost_price,initial_levels,current_spots,strike,barrier_pct,coupon,initial_fixing_date,maturity_date,barrier_breached
0,CH1483491150,BRC,European,[ALCON],[CH0432492467],CHF,10,1000,1.00,[59.72],[58.76],[59.72],0.7,0.0400,2025-11-10,2026-11-17,False
1,CH1449111066,MBRC,European,"[ABB, HOLCIM, NOVARTIS, ROCHE]","[CH0012221716, CH0012214059, CH0012005267, CH0...",CHF,5,1000,0.98,"[35.0, 70.0, 90.0, 250.0]","[34.0, 68.0, 92.0, 245.0]","[35.0, 70.0, 90.0, 250.0]",0.7,0.0675,2025-12-30,2026-12-28,True
2,CH1461018793,MBRC,European,"[ABB, LONZA, NESTLE]","[CH0012221716, CH0013841017, CH0038863350]",CHF,1,10000,1.00,"[53.94, 555.2, 72.49]","[53.94, 555.2, 72.49]","[53.94, 555.2, 72.49]",0.7,0.0866,2025-08-19,2026-08-19,False


### 🔹 Product-Level Analytics — `ReverseConvertible` Class

The `ReverseConvertible` class is used to evaluate a single structured product based on its contractual terms, market data, and an optional scenario applied to the underlying assets.

It takes one portfolio row as input and can additionally accept a list of scenario shocks in percentage terms for each underlying. For example:



In [5]:
rc1 = ReverseConvertible(portfolio.iloc[1], [-10, 5, 0, -3])

In this case, the class evaluates the second product in the portfolio and applies scenario shocks of `-10%`, `+5%`, `0%`, and `-3%` to its four underlyings.

The class then derives the key product analytics, including:

- final underlying performances   
- worst-of performance for multi-asset products  
- payoff per unit and total payoff per position 
- total cost, P&L, and return metrics  
- distance to barrier and break-even level  

This provides a transparent product-level framework for analyzing both single-name and multi-asset reverse convertibles under base-case or stressed market scenarios.

In [14]:
summary_df = pd.DataFrame([rc1.summary()])
summary_df = summary_df.round(2)
summary_df.T.rename(columns={0: "value"})

,value
product_type,MBRC
is_multi,True
maturity_date,2026-12-28
days_to_expiry,268
performance,-0.13
barrier_breached,True
worst_underlying,ABB
payoff_per_unit,941.42
total_payoff,4707.08
total_cost,4900.0


If no scenario shocks are provided, the class defaults to zero shocks for all underlyings.  
As a result, final levels are assumed to be equal to current spot levels, which serves as the base-case valuation scenario.


In [181]:
analytics = PortfolioAnalytics(portfolio)

analytics.build_product_analytics()

analytics.total_portfolio_table()

,total_products,total_notional,total_cost,total_payoff,total_pnl,portfolio_return_pct,portfolio_return_pa
0,3,25000,24900.0,26305.714608,1405.714608,0.056454,0.055561


In [113]:
beta_table = pd.DataFrame({
    "isin": [
        "CH0432492467",
        "CH0012221716",
        "CH0012214059",
        "CH0012005267",
        "CH0012032048"
    ],
    "beta": [0.85, 1.10, 0.95, 0.80, 1.05]
})
scenarios = {
    "down_5": -5,
    "down_10": -10,
    "crash": -20,
    "up_10": 10
}

beta_map = dict(zip(beta_table["isin"], beta_table["beta"]))
beta_map

{'CH0432492467': 0.85,
 'CH0012221716': 1.1,
 'CH0012214059': 0.95,
 'CH0012005267': 0.8,
 'CH0012032048': 1.05}

In [167]:
engine = ScenarioEngine(portfolio, beta_map, scenarios)

In [183]:

df = engine.run(-5)
df["portfolio_summary"]

,market_shock,n_products,total_cost,total_payoff,total_pnl,portfolio_return_pct
0,-5,3,24900.0,25070.403632,170.403632,0.006844


In [169]:
portfolio.iloc[1]["underlying_isins"]

['CH0012221716', 'CH0012214059', 'CH0012005267', 'CH0012032048']